# Prueba individual · Sesión 03
### Career Tech Hub · Un caso de datos de principio a fin

**Nombre:** *(escribe aquí tu nombre)*

Trabajas con los mismos datos de la sesión. Lee antes el PDF de la prueba: ahí están las instrucciones y la rúbrica.

- Haz **al menos dos** de las partes A (pandas), B (SQL) y C (Excel o Power BI).
- La **parte D es obligatoria**.
- Este cuaderno tiene que estar en la misma carpeta que el cuaderno de la sesión, junto a la carpeta `datos`.

In [28]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import sqlite3
import pandas as pd

# Las hojas del Excel (para la parte A)
dias = pd.read_excel("electricidad_para_panel.xlsx", sheet_name="dias")
generacion = pd.read_excel("electricidad_para_panel.xlsx", sheet_name="generacion")

# La base de datos (para la parte B)
conexion = sqlite3.connect("electricidad.db")

def sql(consulta):
    # Lanza una consulta y devuelve el resultado como tabla de pandas.
    return pd.read_sql(consulta, conexion)

print("Hoja dias:", dias.shape[0], "filas ·", list(dias.columns))
sql("SELECT type AS tipo, name AS nombre FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name")

Hoja dias: 1415 filas · ['fecha', 'dia_semana', 'mes', 'festivo', 'descanso', 't_med', 'demanda_gwh', 'año']


,tipo,nombre
0,table,ciudades
1,table,demanda
2,table,festivos
3,table,generacion
4,table,predicciones
5,table,tiempo
6,view,tabla_dias
7,view,temperatura_nacional


In [30]:
import os
from pathlib import Path

print("Dónde está VS Code ahora:", os.getcwd())
print("Archivos aquí:", os.listdir())

for carpeta in [Path("."), Path(".."), Path.home() / "Downloads", Path.home() / "Documents", Path.home() / "Desktop"]:
    if carpeta.exists():
        for f in carpeta.rglob("*electricidad*"):
            print("ENCONTRADO ->", f.resolve())

Dónde está VS Code ahora: c:\Users\User\Desktop\Bootcamp\Electricidad\datos
Archivos aquí: ['crudo', 'Electricidad 2026', 'electricidad.db', 'electricidad_para_panel.xlsx', 'Prueba_individual_plantilla.ipynb']
ENCONTRADO -> C:\Users\User\Desktop\Bootcamp\Electricidad\datos\Electricidad 2026
ENCONTRADO -> C:\Users\User\Desktop\Bootcamp\Electricidad\datos\electricidad.db
ENCONTRADO -> C:\Users\User\Desktop\Bootcamp\Electricidad\datos\electricidad_para_panel.xlsx
ENCONTRADO -> C:\Users\User\Desktop\Bootcamp\Electricidad\datos\Electricidad 2026
ENCONTRADO -> C:\Users\User\Desktop\Bootcamp\Electricidad\datos\electricidad.db
ENCONTRADO -> C:\Users\User\Desktop\Bootcamp\Electricidad\datos\electricidad_para_panel.xlsx
ENCONTRADO -> C:\Users\User\Desktop\Bootcamp\Electricidad
ENCONTRADO -> C:\Users\User\Desktop\Bootcamp\Electricidad\datos\Electricidad 2026
ENCONTRADO -> C:\Users\User\Desktop\Bootcamp\Electricidad\datos\electricidad.db
ENCONTRADO -> C:\Users\User\Desktop\Bootcamp\Electricidad\da

Columnas de las tablas de la base de datos:

| Tabla | Columnas |
|---|---|
| `demanda` | `fecha`, `demanda_gwh` |
| `generacion` | `fecha`, `tecnologia`, `tipo`, `gwh` |
| `tiempo` | `fecha`, `ciudad`, `t_max`, `t_min`, `t_med`, `lluvia_mm`, `viento_kmh`, `sol_mj` |
| `ciudades` | `ciudad`, `latitud`, `longitud`, `habitantes` |
| `festivos` | `fecha`, `festivo` |
| `temperatura_nacional` (vista) | `fecha`, `t_med` |
| `tabla_dias` (vista) | `fecha`, `demanda_gwh`, `t_med`, `festivo`, `demanda_ayer`, `demanda_hace_7` |

# Parte A · pandas

Usa la tabla `dias`.

In [31]:
dias

,fecha,dia_semana,mes,festivo,descanso,t_med,demanda_gwh,año
0,2023-01-01,domingo,1,1,True,10.24,506.425906,2023
1,2023-01-02,lunes,1,0,False,10.25,621.645824,2023
2,2023-01-03,martes,1,0,False,9.20,677.052137,2023
3,2023-01-04,miércoles,1,0,False,7.79,689.160950,2023
4,2023-01-05,jueves,1,0,False,7.32,671.291045,2023
...,...,...,...,...,...,...,...,...
1410,2026-11-11,miércoles,11,0,False,16.12,688.687550,2026
1411,2026-11-12,jueves,11,0,False,16.39,698.606940,2026
1412,2026-11-13,viernes,11,0,False,15.05,707.449750,2026
1413,2026-11-14,sábado,11,0,True,14.13,615.539670,2026


**A1.** ¿Cuántos días superaron los 800 GWh de demanda? ¿Cuántos fueron en cada año?

In [32]:
dias_800_gwh= dias[dias["demanda_gwh"]>800]
print("Días que superaron los 800GWh:", len(dias_800_gwh))
print()
print("Días que superaron los 800GWh por año:")
dias_800_gwh["año"].value_counts().sort_index()

Días que superaron los 800GWh: 115

Días que superaron los 800GWh por año:


año
2023    16
2024    10
2025    25
2026    64
Name: count, dtype: int64

**A2.** Haz una tabla con la demanda media de cada año, separando los días laborables de los días de descanso (columna `descanso`). ¿Qué porcentaje menos se gasta en un día de descanso, cada año?

In [33]:
tabla_demanda = dias.pivot_table(index="año",columns="descanso",values="demanda_gwh",aggfunc="mean")

tabla_demanda = tabla_demanda.rename(columns={False: "Laborable", True: "Descanso"})

tabla_demanda["Porcentaje_menos (%)"] = ((tabla_demanda["Laborable"] - tabla_demanda["Descanso"]) / tabla_demanda["Laborable"] * 100).round(2)

tabla_demanda

descanso,Laborable,Descanso,Porcentaje_menos (%)
año,,,
2023,707.720644,605.501375,14.44
2024,710.062423,612.015598,13.81
2025,732.488655,633.228393,13.55
2026,749.862854,643.033811,14.25


**A3.** ¿Cuál fue el mes con mayor demanda media de los cuatro años (di el mes y el año)? ¿Y el de menor?

In [50]:
medias_gasto=dias.groupby(["año", "mes"])["demanda_gwh"].mean()
anio_max, mes_max = medias_gasto.idxmax()
anio_min, mes_min = medias_gasto.idxmin()
print("Año", anio_max, "del", mes_max, "es cuando mayor demanda hay.")
print("Año", anio_min, "del", mes_min, "es cuando menor demanda hay.")

valor_max = medias_gasto.max()
valor_min = medias_gasto.min()
print("Valor máximo:", valor_max,"y valor mínimo:", valor_min)

Año 2026 del 7 es cuando mayor demanda hay.
Año 2023 del 4 es cuando menor demanda hay.
Valor máximo: 814.8263403225806 y valor mínimo: 610.55781


# Parte B · SQL

Usa la función `sql("...")` con la base de datos.

In [35]:
sql("SELECT type AS tipo, name AS nombre FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name")

,tipo,nombre
0,table,ciudades
1,table,demanda
2,table,festivos
3,table,generacion
4,table,predicciones
5,table,tiempo
6,view,tabla_dias
7,view,temperatura_nacional


In [36]:
sql("""
SELECT * 
FROM demanda
"""
)

,fecha,demanda_gwh
0,2023-01-01,506.425906
1,2023-01-02,621.645824
2,2023-01-03,677.052137
3,2023-01-04,689.160950
4,2023-01-05,671.291045
...,...,...
1410,2026-11-11,688.687550
1411,2026-11-12,698.606940
1412,2026-11-13,707.449750
1413,2026-11-14,615.539670


In [37]:
sql("""
SELECT * 
FROM temperatura_nacional
"""
)

,fecha,t_med
0,2023-01-01,10.24
1,2023-01-02,10.25
2,2023-01-03,9.20
3,2023-01-04,7.79
4,2023-01-05,7.32
...,...,...
1411,2026-11-12,16.39
1412,2026-11-13,15.05
1413,2026-11-14,14.13
1414,2026-11-15,13.97


**B1.** Saca los cinco días más calurosos (vista `temperatura_nacional`) junto con su demanda. ¿Son también días de mucha demanda? Explica por qué en una o dos frases.

In [38]:
sql("""
SELECT temp.t_med, temp.fecha,dem.demanda_gwh
FROM temperatura_nacional temp
JOIN demanda dem ON dem.fecha=temp.fecha
ORDER BY t_med DESC
LIMIT 5
"""
)

# Temperaturas similares y diferencia entre demandas. Al ser dias de vacaciones o operación salida o retorno de ellas, casualidad esos dias(11,16,17 de agosto) tienen menos demanda que el 23. 


,t_med,fecha,demanda_gwh
0,31.81,2025-08-16,712.442720
1,31.79,2025-08-17,698.786118
2,31.61,2023-08-23,805.114493
3,31.49,2025-08-11,770.154701
4,31.43,2025-06-29,707.452281


In [39]:
sql("""
SELECT *
FROM generacion
GROUP BY tecnologia
"""
)

,fecha,tecnologia,tipo,gwh
0,2023-01-01,Carbón,No-Renovable,10.982254
1,2023-01-01,Ciclo combinado,No-Renovable,65.377808
2,2023-01-01,Cogeneración,No-Renovable,19.340669
3,2023-01-01,Eólica,Renovable,161.362747
4,2023-01-01,Hidroeólica,Renovable,0.004500
5,2023-01-01,Hidráulica,Renovable,78.179148
6,2023-01-01,Motores diésel,No-Renovable,5.630689
7,2023-01-01,Nuclear,No-Renovable,155.560191
8,2023-01-01,Otras renovables,Renovable,5.311810
9,2023-01-01,Residuos no renovables,No-Renovable,3.318656


**B2.** Para cada año, calcula la generación solar fotovoltaica en TWh y qué porcentaje supone de toda la generación del año.

In [52]:
sql("""

SELECT
    strftime('%Y', fecha) AS año,
    SUM(CASE WHEN tecnologia LIKE '%Solar fotovoltaica%' THEN gwh ELSE 0 END) / 1000.0 AS twh_solar,
    (SUM(CASE WHEN tecnologia LIKE '%Solar fotovoltaica%' THEN gwh ELSE 0 END) * 100.0) / SUM(gwh) AS porcentaje_solar
FROM generacion
GROUP BY año
ORDER BY año;
"""
)


,año,twh_solar,porcentaje_solar
0,2023,37.453800,14.207912
1,2024,44.598990,16.994313
2,2025,50.291509,18.462012
3,2026,55.410234,22.400034


In [41]:
sql("""
SELECT * 
FROM tiempo
"""
)

,fecha,ciudad,t_max,t_min,t_med,lluvia_mm,viento_kmh,sol_mj
0,2023-01-01,Madrid,14.6,3.6,8.6,0.0,12.3,7.52
1,2023-01-02,Madrid,11.4,6.4,8.9,1.1,10.2,5.49
2,2023-01-03,Madrid,11.5,3.1,6.9,0.0,9.2,8.85
3,2023-01-04,Madrid,11.1,0.5,5.0,0.0,7.5,9.35
4,2023-01-05,Madrid,10.9,0.8,4.8,0.0,9.5,9.35
...,...,...,...,...,...,...,...,...
8491,2026-11-12,Bilbao,18.2,13.6,15.1,18.3,19.5,6.33
8492,2026-11-13,Bilbao,13.8,11.4,13.0,3.6,7.7,2.65
8493,2026-11-14,Bilbao,17.0,10.0,12.9,11.3,8.8,6.72
8494,2026-11-15,Bilbao,18.0,10.2,14.0,3.1,27.2,3.82


**B3.** ¿Cuántos días de lluvia (1 mm o más) ha tenido cada ciudad? Ordena de más a menos.

In [42]:
sql("""
SELECT ciudad, COUNT(*) AS dias, SUM(CASE WHEN lluvia_mm >= 1 THEN 1 ELSE 0 END) AS dias_lluvia
FROM tiempo
GROUP BY ciudad
ORDER BY dias_lluvia DESC
""")

,ciudad,dias,dias_lluvia
0,Bilbao,1416,611
1,Barcelona,1416,350
2,Madrid,1416,305
3,Zaragoza,1416,269
4,Sevilla,1416,239
5,Valencia,1416,218


In [44]:
sql("""
SELECT * 
FROM demanda
"""
)

,fecha,demanda_gwh
0,2023-01-01,506.425906
1,2023-01-02,621.645824
2,2023-01-03,677.052137
3,2023-01-04,689.160950
4,2023-01-05,671.291045
...,...,...
1410,2026-11-11,688.687550
1411,2026-11-12,698.606940
1412,2026-11-13,707.449750
1413,2026-11-14,615.539670


**Reto (opcional).** Con `LAG`, saca los tres días en que la demanda más subió respecto al día anterior. ¿Qué pasó en el primero?

In [46]:
sql("""


WITH demanda_diaria AS (
    SELECT
        fecha,
        SUM(demanda_gwh) AS demanda_total
    FROM demanda
    GROUP BY fecha
),
con_anterior AS (
    SELECT
        fecha,
        demanda_total,
        LAG(demanda_total) OVER (ORDER BY fecha) AS demanda_ayer
    FROM demanda_diaria
)
SELECT
    fecha,
    demanda_total,
    demanda_ayer,
    demanda_total - demanda_ayer AS subida
FROM con_anterior
WHERE demanda_ayer IS NOT NULL
ORDER BY subida DESC
LIMIT 3;

""")


,fecha,demanda_total,demanda_ayer,subida
0,2025-04-29,590.008854,403.297490,186.711364
1,2026-01-07,849.726680,681.283230,168.443450
2,2023-07-10,778.800770,628.912612,149.888158


# Parte C · Excel o Power BI

Esta parte no se hace en el cuaderno. Abre `datos/electricidad_para_panel.xlsx` en Excel o cárgalo en Power BI.

**C1.** Haz una tabla dinámica (en Power BI, una matriz) con la demanda media: los meses en filas y los años en columnas. Hay una celda que no se puede comparar con las demás. ¿Cuál es y por qué?

**C2.** Haz un gráfico de la generación de 2025 por tecnología (hoja `generacion`), ordenado de mayor a menor. Ponle un título que diga la conclusión, no solo el tema.

Entrega una captura de cada una. Puedes pegarlas en un documento o guardarlas como imagen.

# Parte D · Explica (obligatoria)

Contesta con tus palabras. Frases cortas.

**D1.** Elige una decisión que hayas tenido que tomar en esta prueba y explica por qué la tomaste (4 a 6 líneas).

*En B2 decidí meter el cambio de GWh a TWh (/ 1000) directamente en la fórmula para tener el dato ya listo y limpio. También usé CASE WHEN para sumar bien la solar fotovoltaica dentro del SELECT y calcular su peso sobre el total anual. En B3 apliqué un CASE WHEN >= 1 para contar los días de lluvia sin tener que hacer subconsultas. Por último, en el reto usé WITH y LAG para comparar cada día con el anterior, lo que me permitió detectar el salto del 29/04/2025 tras el apagón del 28.*

**D2.** Cuenta tu resultado más interesante a alguien que no sabe de datos (3 líneas).

*Lo que mas me ha llamado la atención es ver como hemos aumentado los picos de consumo de energía en España este 2026 ya que sumando los 3 anteriores años los días de  consumo de energia de más de 800 gwh es inferior.64 dias llevamos este 2026 por 51 entre 2023-2025. Como dato curioso el día que mas subió la demanda respecto al día anterio fue el día post-apagón 29/04/2025. Y otro dato interesante en cúanto a la lluvia que Bilbao casi duplica a Barcelona en días de lluvia en estos 4 últimos años(611 vS 350)*

**D3.** Di un límite de tu análisis, o algo que comprobarías si tuvieras más tiempo.

*Si tuviera más tiempo, analizaría tablas que no he tocado a fondo como predicciones y festivos para comparar datos y sacar más conclusiones. Me interesa ver en qué se desvía predicciones frente a la demanda real y cómo cambia el consumo en los días festivos.*

**D4.** Si has usado una IA, di para qué y cómo comprobaste que el resultado era correcto. Si no la has usado, dilo.

*Obviamente he utilizado Gémini porque muchas veces no me sale a la primera una consulta. Donde más me ha hecho falta es en los días de descanso (.pivot_table), en la demanda media por año y mes (.idxmax() y .idxmin()) y en el reto para usar LAG. Para comprobar que el resultado era correcto, revisé las tablas que salían en el cuaderno y contrasté que los datos cuadraran con la lógica y fechas reales, como el salto tras el apagón del 28 de abril de 2025.*